# ReasonCritic-7B Fine-Tuning with Unsloth (DPO)

Fine-tune `Qwen/Qwen2.5-Coder-7B-Instruct` on code critique DPO pairs.

**Runtime**: GPU T4 (Colab free) or A100 (Colab Pro) | **Time**: ~45 min on T4 | **VRAM**: ~12GB

Uses 4-bit QLoRA + DPO training for preference alignment.

In [ ]:
%%capture
!pip install unsloth
!pip uninstall -y transformers && pip install transformers==4.46.3

from unsloth import FastLanguageModel
import torch

In [ ]:
# Configuration
MODEL_NAME = "Qwen/Qwen2.5-Coder-7B-Instruct"
HF_REPO = "fableforge-ai/ReasonCritic-7B"
MAX_SEQ_LENGTH = 2048
LORA_R = 16

# Load model with 4-bit QLoRA
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    dtype=None,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

print(f"Model loaded: {MODEL_NAME}")
print(f"LoRA rank: {LORA_R}")

In [ ]:
# Load DPO data
from datasets import Dataset
import json

TRAIN_PATH = "reasoncritic_dpo_train.jsonl"
VAL_PATH = "reasoncritic_dpo_val.jsonl"

def load_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f]

raw_train = load_jsonl(TRAIN_PATH)
raw_val = load_jsonl(VAL_PATH)

# Format for DPO training
SYSTEM_PROMPT = "You are ReasonCritic, an expert code reviewer who identifies bugs, security issues, and design flaws. Provide thorough, accurate critiques with specific fixes."

def format_dpo(example):
    return {
        "prompt": f"{SYSTEM_PROMPT}\n\n{example['prompt']}",
        "chosen": example["chosen"],
        "rejected": example["rejected"],
    }

train_data = [format_dpo(ex) for ex in raw_train]
val_data = [format_dpo(ex) for ex in raw_val]

train_dataset = Dataset.from_list(train_data)
val_dataset = Dataset.from_list(val_data)

print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}")

In [ ]:
# DPO Training
from trl import DPOConfig, DPOTrainer

training_args = DPOConfig(
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_ratio=0.1,
    num_train_epochs=3,
    learning_rate=5e-5,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=10,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    seed=42,
    output_dir="outputs",
    report_to="none",
    beta=0.1,  # DPO beta (KL penalty)
    max_length=MAX_SEQ_LENGTH,
)

trainer = DPOTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    args=training_args,
)

trainer_stats = trainer.train()
print(f"Training time: {trainer_stats.metrics['train_runtime']:.1f}s")

In [ ]:
# Test inference
FastLanguageModel.for_inference(model)

test_prompt = SYSTEM_PROMPT + "\n\nIs this code thread-safe? class Counter:\n    def __init__(self):\n        self.count = 0\n    def increment(self):\n        self.count += 1"

messages = [{"role": "user", "content": test_prompt}]
inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")

outputs = model.generate(
    input_ids=inputs,
    max_new_tokens=256,
    temperature=0.3,
    top_p=0.9,
    use_cache=True,
)
print(tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True))

In [ ]:
# Save LoRA adapters
model.save_pretrained("reasoncritic-lora")
tokenizer.save_pretrained("reasoncritic-lora")
print("LoRA adapters saved to ./reasoncritic-lora")

In [ ]:
# Merge and save full model
model.save_pretrained_merged("reasoncritic-merged", tokenizer)
print("Merged model saved to ./reasoncritic-merged")

In [ ]:
# Push to HuggingFace
# from huggingface_hub import login
# login(token="your_token_here")
# model.push_to_hub_merged(HF_REPO, tokenizer)
# print(f"Model pushed to https://huggingface.co/{HF_REPO}")